# 13 — Finalni sažetak: master tablica i 2×2 faktorijal

Faza 5: **F5.1**. Ovaj notebook preuzima ulogu starog `06_summary` — **ne
preračunava** rezultate, nego učitava **zamrznute spremljene panele i tablice**
faza 1–4 i konsolidira ih u dvije izlazne tablice:

- `outputs/tables/13_master_table.csv` — svih 37 portfelja završne usporedbe ×
  sve metrike (FF5 bete, koncentracija stila [CI], godišnja volatilnost, Sharpe
  bruto/neto, maksimalni pad, obrtaj, efektivni N, DSR, MCS članstvo, `n_months`);
- `outputs/tables/13_factorial_2x2.csv` — 2×2 faktorijal {prostor hijerarhije} ×
  {bez/sa overlay ograničenjem} na primarnom ishodu (koncentracija stila) i
  ostvarenoj volatilnosti, prosječeno preko HRP/HERC/NCO.

**Izvori (sve već persistirano, bez novog backtesta):** koncentracija stila i
volatilnost iz `12_frontier_points` (37); Sharpe i DSR iz `12_dsr` (37); MCS
članstvo iz `12_mcs_final` (37×2); FF5 bete iz `10_attribution_full_sample` (10
dijagnostičkih portfelja Faze 2); bootstrap CI koncentracije stila iz
`10_bootstrap_ci` (6). Maksimalni pad, obrtaj i efektivni N su deskriptivne
statistike izračunate iz **spremljenih** panela prinosa/težina (nije ponovni
backtest); podudarnost s `09_replication_summary` provjerava se u 13.5.

Korelacijska HRP baza u 2×2 je **`hrp_corr_ward`** (krak kontrolirane usporedbe,
K1); `hrp_corr_single` (jednostruka veza) ostaje u master tablici kao
replikacijski red, ali nije u uzročnom 2×2.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation import max_drawdown, turnover_per_window
from src.utils import TABLES_DIR, ensure_dirs, set_seed

ensure_dirs()
set_seed()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 13.1 Učitavanje spremljenih tablica i panela

Učitavaju se isključivo zamrznuti izlazi faza 1–4. Nijedan alokator,
bootstrap ni regresija se ne pokreće ponovno.

In [2]:
# Statističke tablice (bez ponovnog računanja) ------------------------------
frontier = pd.read_csv(TABLES_DIR / "12_frontier_points.csv")        # 37: ann_vol, style_conc, n_months
dsr = pd.read_csv(TABLES_DIR / "12_dsr.csv")                         # 37: sharpe, dsr, n_trials
mcs = pd.read_csv(TABLES_DIR / "12_mcs_final.csv")                   # 37 x {gross, net}
attrib = pd.read_csv(TABLES_DIR / "10_attribution_full_sample.csv")  # 10: FF5 bete
boot = pd.read_csv(TABLES_DIR / "10_bootstrap_ci.csv")              # bootstrap CI koncentracije stila

# Panel prinosa (spoj zamrznutih panela faza 1/3/4 + benchmark obitelj) -------
_ret_files = [
    "05_port_returns_panel_extended",
    "09_port_returns_panel_hierarchical",
    "11_port_returns_panel_factor",
    "12_port_returns_panel_overlay",
]
ret_panel = None
for name in _ret_files:
    part = pd.read_csv(TABLES_DIR / f"{name}.csv", index_col=0, parse_dates=True)
    ret_panel = part if ret_panel is None else ret_panel.join(part, how="outer")

# Panel težina (dugi format) za obrtaj i efektivni N -------------------------
_w_files = [
    "weights_panel",
    "05_weights_panel_factor_neutral",
    "09_weights_panel_hierarchical",
    "11_weights_panel_factor",
    "12_weights_panel_overlay",
]
w_panel = pd.concat(
    [pd.read_csv(TABLES_DIR / f"{name}.csv") for name in _w_files],
    ignore_index=True,
)

print("portfelja (frontier):", frontier["portfolio"].nunique())
print("panel prinosa:", ret_panel.shape, "| panel težina portfelja:", w_panel["portfolio"].nunique())
assert frontier["portfolio"].nunique() == 37
assert ret_panel.shape[1] == 37 and w_panel["portfolio"].nunique() == 37

portfelja (frontier): 37
panel prinosa: (156, 37) | panel težina portfelja: 37


## 13.2 Deskriptivne statistike iz spremljenih panela

Maksimalni pad (iz bruto panela prinosa), jednostrani obrtaj
(`turnover_per_window`, prosjek kroz prozore uključujući prvi = 1.0) i
efektivni broj pozicija `N_eff = 1 / Σ_i w_i²` (prosjek kroz prozore). Sve su
čiste funkcije već spremljenih panela — ne pokreću backtest.

In [3]:
# Maksimalni pad po portfelju (bruto, isto kao 09_replication_summary)
mdd = pd.Series(
    {col: max_drawdown(ret_panel[col]) for col in ret_panel.columns},
    name="max_drawdown",
)

# Prosječni jednostrani obrtaj po portfelju (uklj. prvi prozor = 1.0)
turnover = (
    turnover_per_window(w_panel).groupby("portfolio")["turnover"].mean().rename("turnover")
)

# Efektivni N = prosjek kroz prozore od 1 / Σ w_i²
_hhi = w_panel.groupby(["portfolio", "train_window"])["weight"].apply(
    lambda w: float((w.to_numpy(dtype=float) ** 2).sum())
)
eff_n = (1.0 / _hhi).groupby("portfolio").mean().rename("eff_n")

print(mdd.round(4).head())
print(turnover.round(4).head())
print(eff_n.round(1).head())

equal_weight           -0.2678
min_var                -0.1900
factor_neutral_e0      -0.1915
factor_neutral_e0.05   -0.1862
factor_neutral_e0.1    -0.1839
Name: max_drawdown, dtype: float64
portfolio
equal_weight            0.1213
factor_neutral_e0       0.5563
factor_neutral_e0.05    0.5441
factor_neutral_e0.1     0.5370
factor_neutral_e0.15    0.5340
Name: turnover, dtype: float64
portfolio
equal_weight            410.8
factor_neutral_e0        30.3
factor_neutral_e0.05     30.6
factor_neutral_e0.1      30.6
factor_neutral_e0.15     30.6
Name: eff_n, dtype: float64


## 13.3 Master tablica

Spine = 37 portfelja iz `12_frontier_points`. Lijevim spajanjem po imenu
portfelja dodaju se metrike iz ostalih tablica; gdje statistika nije
persistirana za neku varijantu (FF5 bete i CI postoje samo za dijagnostički
skup Faze 2) ćelija ostaje `NaN` — bez naknadnog računanja.

In [4]:
def _classify(name: str) -> tuple[str, str, float]:
    """(prostor, overlay-oznaka, epsilon) iz imena portfelja."""
    if name in ("equal_weight", "min_var"):
        return "benchmark", "none", np.nan
    m = re.match(r"factor_neutral_e([0-9.]+)$", name)
    if m:
        return "benchmark", "factor_neutral", float(m.group(1))
    space = "factor" if "factor" in name else "correlation"
    ov = re.search(r"_ov_e([0-9.]+)$", name)
    return space, ("overlay" if ov else "base"), (float(ov.group(1)) if ov else np.nan)


def _allocator(name: str) -> str:
    for a in ("hrp", "herc", "nco"):
        if name.startswith(a):
            return a
    if name.startswith("factor_neutral"):
        return "min_var_fn"
    return name


master = frontier[["portfolio", "ann_vol", "style_concentration", "n_months"]].copy()
_meta = master["portfolio"].apply(_classify)
master["space"] = [m[0] for m in _meta]
master["overlay"] = [m[1] for m in _meta]
master["epsilon"] = [m[2] for m in _meta]
master["allocator"] = master["portfolio"].apply(_allocator)

# FF5 bete (10 dijagnostičkih portfelja Faze 2)
master = master.merge(
    attrib[["portfolio", "beta_mkt", "beta_smb", "beta_hml", "beta_rmw", "beta_cma", "alpha", "r_squared"]],
    on="portfolio", how="left",
)
# Bootstrap CI koncentracije stila (6 portfelja)
_style_ci = (
    boot[boot["metric"] == "style_concentration"][["portfolio", "ci_low", "ci_high"]]
    .rename(columns={"ci_low": "style_ci_low", "ci_high": "style_ci_high"})
)
master = master.merge(_style_ci, on="portfolio", how="left")
# Sharpe + DSR (37)
master = master.merge(
    dsr[["portfolio", "sharpe_gross", "sharpe_net", "dsr_gross", "dsr_net", "n_trials"]],
    on="portfolio", how="left",
)
# MCS članstvo bruto/neto (37)
_mcs_g = mcs[mcs["returns_type"] == "gross"][["portfolio", "in_mcs", "p_value"]].rename(
    columns={"in_mcs": "in_mcs_gross", "p_value": "mcs_p_gross"})
_mcs_n = mcs[mcs["returns_type"] == "net"][["portfolio", "in_mcs", "p_value"]].rename(
    columns={"in_mcs": "in_mcs_net", "p_value": "mcs_p_net"})
master = master.merge(_mcs_g, on="portfolio", how="left").merge(_mcs_n, on="portfolio", how="left")
# Deskriptivne statistike (37)
master = (
    master.merge(mdd, left_on="portfolio", right_index=True)
    .merge(turnover, left_on="portfolio", right_index=True)
    .merge(eff_n, left_on="portfolio", right_index=True)
)

_cols = [
    "portfolio", "allocator", "space", "overlay", "epsilon", "n_months",
    "beta_mkt", "beta_smb", "beta_hml", "beta_rmw", "beta_cma", "alpha", "r_squared",
    "style_concentration", "style_ci_low", "style_ci_high",
    "ann_vol", "sharpe_gross", "sharpe_net", "max_drawdown", "turnover", "eff_n",
    "dsr_gross", "dsr_net", "in_mcs_gross", "in_mcs_net", "mcs_p_gross", "mcs_p_net", "n_trials",
]
master = (
    master[_cols]
    .sort_values(["space", "allocator", "overlay", "epsilon"])
    .reset_index(drop=True)
)

master.to_csv(TABLES_DIR / "13_master_table.csv", index=False)
print("master:", master.shape, "| portfelja:", master["portfolio"].nunique())
print("FF5 bete ne-null:", int(master["beta_smb"].notna().sum()),
      "| CI stila ne-null:", int(master["style_ci_low"].notna().sum()))
master[["portfolio", "space", "overlay", "style_concentration", "ann_vol",
        "sharpe_gross", "dsr_gross", "in_mcs_gross", "eff_n"]].round(4)

master: (37, 29) | portfelja: 37
FF5 bete ne-null: 10 | CI stila ne-null: 6


,portfolio,space,overlay,style_concentration,ann_vol,sharpe_gross,dsr_gross,in_mcs_gross,eff_n
0,equal_weight,benchmark,none,0.5948,0.1559,0.7815,0.6791,False,410.8462
1,min_var,benchmark,none,0.7761,0.1270,0.7277,0.6192,True,30.7751
2,factor_neutral_e0,benchmark,factor_neutral,0.5255,0.1307,0.7941,0.6932,True,30.2591
3,factor_neutral_e0.05,benchmark,factor_neutral,0.5906,0.1286,0.7824,0.6780,True,30.5507
4,factor_neutral_e0.1,benchmark,factor_neutral,0.6688,0.1271,0.7676,0.6613,True,30.5544
5,factor_neutral_e0.15,benchmark,factor_neutral,0.7261,0.1269,0.7550,0.6481,True,30.5907
6,herc_corr,correlation,base,0.5739,0.1431,0.7374,0.6086,False,96.5667
7,herc_corr_ov_e0,correlation,overlay,0.3865,0.1453,0.7173,0.5746,False,74.2537
8,herc_corr_ov_e0.05,correlation,overlay,0.4295,0.1439,0.7112,0.5698,False,83.5419
9,herc_corr_ov_e0.1,correlation,overlay,0.4670,0.1424,0.7134,0.5757,False,90.5950


## 13.4 2×2 faktorijalna tablica

Faktorijal {prostor: korelacijski, faktorski} × {overlay: bez (baza),
sa (ε = 0, najstrože)} na hijerarhijskim alokatorima. Ćelija = prosjek preko
HRP/HERC/NCO. Korelacijski HRP = `hrp_corr_ward` (K1). Primarni ishod je
koncentracija stila; ostvarena volatilnost je sekundarna (cijena ograničenja).

In [5]:
_factorial_names = {
    ("correlation", "base"): ["hrp_corr_ward", "herc_corr", "nco_corr"],
    ("correlation", "overlay"): ["hrp_corr_ov_e0", "herc_corr_ov_e0", "nco_corr_ov_e0"],
    ("factor", "base"): ["hrp_factor", "herc_factor", "nco_factor"],
    ("factor", "overlay"): ["hrp_factor_ov_e0", "herc_factor_ov_e0", "nco_factor_ov_e0"],
}
_idx = master.set_index("portfolio")
rows = []
for (space, overlay), names in _factorial_names.items():
    sub = _idx.loc[names]
    rows.append({
        "space": space,
        "overlay": overlay,
        "mean_style_concentration": float(sub["style_concentration"].mean()),
        "mean_ann_vol": float(sub["ann_vol"].mean()),
        "n_allocators": len(names),
    })
factorial = pd.DataFrame(rows)
factorial.to_csv(TABLES_DIR / "13_factorial_2x2.csv", index=False)

# Glavni učinci (overlay − baza) po prostoru
for space in ("correlation", "factor"):
    b = factorial.query("space == @space and overlay == 'base'")["mean_style_concentration"].iloc[0]
    o = factorial.query("space == @space and overlay == 'overlay'")["mean_style_concentration"].iloc[0]
    print(f"{space:12s}: koncentracija stila baza={b:.3f} -> overlay(ε=0)={o:.3f} (Δ={o - b:+.3f})")
factorial.round(4)

correlation : koncentracija stila baza=0.647 -> overlay(ε=0)=0.435 (Δ=-0.212)
factor      : koncentracija stila baza=0.660 -> overlay(ε=0)=0.509 (Δ=-0.151)


,space,overlay,mean_style_concentration,mean_ann_vol,n_allocators
0,correlation,base,0.6468,0.1339,3
1,correlation,overlay,0.4351,0.1375,3
2,factor,base,0.6597,0.1376,3
3,factor,overlay,0.5092,0.1403,3


## 13.5 Provjera podudarnosti (spot-uzorak)

Brojevi master tablice moraju se podudarati s izvornim tablicama faza 1–4.
Provjeravaju se: volatilnost/stil (`12_frontier_points`), Sharpe/DSR (`12_dsr`),
MCS (`12_mcs_final`), bete (`10_attribution_full_sample`), te izračunate
deskriptivne statistike (MDD/obrtaj) naspram `09_replication_summary`.

In [6]:
rep = pd.read_csv(TABLES_DIR / "09_replication_summary.csv").set_index("portfolio")
mi = master.set_index("portfolio")

# 1) MDD i obrtaj izračunati ovdje == zamrznuti 09_replication_summary (10 preklopa)
max_dmdd = max(abs(mi.loc[p, "max_drawdown"] - rep.loc[p, "max_drawdown"]) for p in rep.index)
max_dto = max(abs(mi.loc[p, "turnover"] - rep.loc[p, "turnover"]) for p in rep.index)
assert max_dmdd < 1e-9 and max_dto < 1e-9, (max_dmdd, max_dto)

# 2) Volatilnost i stil == 12_frontier_points
fp = frontier.set_index("portfolio")
assert (mi["ann_vol"] - fp["ann_vol"]).abs().max() < 1e-12
assert (mi["style_concentration"] - fp["style_concentration"]).abs().max() < 1e-12

# 3) Sharpe/DSR == 12_dsr ; MCS == 12_mcs_final
d = dsr.set_index("portfolio")
assert (mi["sharpe_gross"] - d["sharpe_gross"]).abs().max() < 1e-12
assert (mi["dsr_gross"] - d["dsr_gross"]).abs().max() < 1e-12
assert int(mi["in_mcs_gross"].sum()) == int(mcs.query("returns_type=='gross' and in_mcs").shape[0])

# 4) Bete == 10_attribution_full_sample (10 portfelja)
a = attrib.set_index("portfolio")
assert (mi.loc[a.index, "beta_rmw"] - a["beta_rmw"]).abs().max() < 1e-12

print(f"MDD/obrtaj naspram 09_replication_summary: max |Δ| MDD={max_dmdd:.2e}, obrtaj={max_dto:.2e}")
print("Sve spot-provjere prolaze: master tablica je dosljedna izvornim tablicama faza 1–4.")
print("\nLijek (overlay ε=0) na primarnom ishodu, prosjek preko HRP/HERC/NCO:")
print(factorial[["space", "overlay", "mean_style_concentration", "mean_ann_vol"]].round(4).to_string(index=False))

MDD/obrtaj naspram 09_replication_summary: max |Δ| MDD=2.78e-16, obrtaj=2.78e-15
Sve spot-provjere prolaze: master tablica je dosljedna izvornim tablicama faza 1–4.

Lijek (overlay ε=0) na primarnom ishodu, prosjek preko HRP/HERC/NCO:
      space overlay  mean_style_concentration  mean_ann_vol
correlation    base                    0.6468        0.1339
correlation overlay                    0.4351        0.1375
     factor    base                    0.6597        0.1376
     factor overlay                    0.5092        0.1403


---

**Sažetak F5.1.** `13_master_table.csv` (37 portfelja × 27 stupaca, ≥ 20) i
`13_factorial_2x2.csv` konsolidiraju zamrznute rezultate faza 1–4 bez
preračunavanja; spot-provjere u 13.5 potvrđuju podudarnost s izvornim tablicama.
2×2 faktorijal pokazuje da overlay ograničenje smanjuje koncentraciju stila u
**oba** prostora uz zanemarivu promjenu volatilnosti — kvantificirana potpora
za H3 iz F4.6.